# Phase 2 — Negative Review Classification (Canonical Notebook)

**Status:** starter workflow; no model results or final decisions are recorded yet.

This is the team's single canonical classification notebook. It predicts whether a delivered order will receive a **negative review (1–2 stars)** immediately after customer delivery and before review submission. It consumes the Phase 1 processed analytical CSVs, not the dashboard Parquet. See `docs/phase2-classification-workflow.md` for full contracts, design decisions and workstream ownership.

Run cells in order after reviewing every TODO. Do not commit claimed metrics without actual execution and team review.

## 1. Problem Definition

**Stakeholder:** customer-experience team. **Decision:** prioritise proactive outreach after delivery. **Target:** 1 when the eventual score is 1–2; otherwise 0 for scores 3–5.

**TODO (team):** confirm the stakeholder action, relative cost of a missed negative review versus unnecessary outreach, and measurable success criteria. Output: an approved problem statement and metric priorities. Do not choose a threshold yet.

## 2. Prediction Point

The prediction is made **after the recorded customer-delivery time and before review submission**. Actual delivery duration and performance are available; review ID, text, creation/answer timestamps and score are unavailable as predictors. The score is used only to construct the retrospective training target.

**TODO (data owner):** obtain team approval for this operational timing. If the prediction point changes, stop and revise the feature contract before modelling.

## 3. Input Data Contract

Inputs are `data/processed/order_level.csv` (one row/order) and `item_level.csv` (one row/order item), produced in Phase 1. `build_classification_dataset()` documents and validates required columns, grain, eligibility, target and output. General Phase 1 cleaning is not repeated here.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    cross_val_predict,
    train_test_split,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classification_data import (
    CATEGORICAL_FEATURES,
    MODEL_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_classification_dataset,
    split_features_target,
)
from src.classification_evaluation import (
    classification_metrics,
    select_f1_threshold,
    threshold_table,
)
from src.classification_model import (
    build_dummy_pipeline,
    build_logistic_pipeline,
    predict_from_artifact,
    save_inference_artifact,
)

RANDOM_SEED = 42
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ORDER_PATH = PROCESSED_DIR / "order_level.csv"
ITEM_PATH = PROCESSED_DIR / "item_level.csv"

order_level = pd.read_csv(ORDER_PATH)
item_level = pd.read_csv(ITEM_PATH)
print(f"order_level: {order_level.shape}; item_level: {item_level.shape}")

## 4. Dataset Construction

Business-specific, deterministic item aggregation stays outside sklearn; it learns no population statistics. Learned imputation, scaling and encoding remain inside the later pipeline.

In [ ]:
classification_data = build_classification_dataset(order_level, item_level)
print(classification_data.shape)
display(classification_data.head())

## 5. Data/Target Audit

Check population size, one-row-per-order grain, missingness, target balance and numeric ranges before any split. The target prevalence describes this historical eligible population, not future performance.

**TODO (data owner):** investigate every unexpected null/range; record exclusion counts by eligibility condition (including reviews answered before or at delivery) and reconcile them to Phase 1. Output: reviewed audit table and written disposition for each issue.

In [ ]:
assert classification_data["order_id"].is_unique

audit = pd.DataFrame({
    "dtype": classification_data.dtypes.astype(str),
    "missing_n": classification_data.isna().sum(),
    "missing_pct": classification_data.isna().mean(),
    "unique_n": classification_data.nunique(dropna=False),
})
display(audit)
display(classification_data[TARGET_COLUMN].value_counts(dropna=False).rename("n").to_frame())
display(classification_data[NUMERIC_FEATURES].describe().T)

## 6. Feature and Leakage Audit

The model matrix is an allow-list, never “all columns except target.” Confirm each field exists at the prediction point. `order_id` is trace-only. Review fields and customer/product/seller identifiers are excluded. Delivery outcomes are allowed only because prediction occurs after delivery.

**TODO (all):** complete a feature-by-feature availability/rationale table and freeze it before fitting. Output: approved allow-list and any proposed removals with reasons.

In [ ]:
feature_audit = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "type": ["numeric" if name in NUMERIC_FEATURES else "categorical" for name in MODEL_FEATURES],
    "available_after_delivery": True,
    "review_required": False,
    "team_rationale": "TODO",
})
display(feature_audit)
assert not any(name.startswith("review_") for name in MODEL_FEATURES)
assert "order_id" not in MODEL_FEATURES

## 7. Protected Stratified Train/Test Split

Create one stratified test set and do not inspect its labels for feature, model, class-weight, hyperparameter or threshold decisions. All such decisions use training folds only.

In [ ]:
X, y = split_features_target(classification_data)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED,
)
print(f"training rows: {len(X_train):,}; protected test rows: {len(X_test):,}")
# Do not calculate test metrics until Section 13.

## 8. Dummy Baseline

Use the same fold-fitted preprocessing as Logistic Regression. Accuracy alone is not a suitable decision metric for an imbalanced outcome.

**TODO (modelling owner):** run stratified CV for the prior dummy, preserve fold-level precision/recall/F1/balanced accuracy/PR-AUC, and explain what “no skill” means. Output: reviewed baseline table, not a manually typed metric.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy",
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
}
dummy_pipeline = build_dummy_pipeline(strategy="prior")

# TODO: uncomment only when ready to execute and review.
# dummy_cv = cross_validate(dummy_pipeline, X_train, y_train, cv=cv, scoring=scoring)
# display(pd.DataFrame(dummy_cv))

## 9. Logistic Regression Pipeline

Median imputation and scaling for numeric features, plus mode imputation and unknown-safe one-hot encoding for categoricals, are fitted inside each fold. The initial classifier is default unweighted Logistic Regression with a documented seed.

In [ ]:
logistic_pipeline = build_logistic_pipeline(random_state=RANDOM_SEED)
logistic_pipeline

## 10. Stratified Cross Validation

**TODO (modelling owner):** evaluate the default pipeline with the identical folds and scoring used for the dummy. Report fold values plus mean/variation and compare to the dummy. Output: reproducible CV result table and concise interpretation; do not touch the protected test set.

In [ ]:
# TODO: uncomment only when ready to execute and review.
# logistic_cv = cross_validate(
#     logistic_pipeline, X_train, y_train, cv=cv, scoring=scoring,
#     return_estimator=True,
# )
# display(pd.DataFrame(logistic_cv))

## 11. Model Refinement

Stay within Logistic Regression. Candidate refinements may include justified class weighting and a small regularisation-strength grid. Change one planned factor at a time and compare on identical training folds.

**TODO (modelling owner):** pre-register the small candidate set, selection metric and tie-break rule. Output: comparison table, rationale and selected configuration. Do not introduce Random Forest or another family here.

In [ ]:
# Example construction only; fitting and selection remain TODO.
weighted_logistic_pipeline = build_logistic_pipeline(
    random_state=RANDOM_SEED,
    class_weight="balanced",
)
# TODO: define a small course-appropriate refinement plan and evaluate on training CV only.

## 12. OOF Probabilities and Threshold Selection

Generate out-of-fold (OOF) probabilities from the selected training-only configuration. Compare thresholds using precision, recall, F1 and balanced accuracy. PR-AUC/ROC-AUC do not change with the threshold. Select based on the stakeholder cost established in Section 1—not test performance.

**TODO (evaluation owner):** replace `selected_pipeline` only after Section 11 review, generate OOF probabilities, produce the threshold table and document the rule. Output: chosen threshold and rationale, both derived without test labels.

In [ ]:
selected_pipeline = clone(logistic_pipeline)  # TODO: replace with reviewed configuration

# TODO: uncomment after the model configuration is frozen.
# oof_probability = cross_val_predict(
#     selected_pipeline, X_train, y_train, cv=cv, method="predict_proba"
# )[:, 1]
# oof_thresholds = threshold_table(y_train, oof_probability)
# display(oof_thresholds)
# selected_threshold = select_f1_threshold(oof_thresholds)  # replace if business rule differs
# print(f"Training-only selected threshold: {selected_threshold:.2f}")

## 13. Final Test Evaluation

Open the protected test set exactly once after freezing the pipeline and threshold. Compare final Logistic Regression with the dummy and report a confusion matrix plus imbalance-aware metrics. Do not return to model selection after viewing the result.

**TODO (evaluation owner):** fit on all training rows, score test probabilities, apply the frozen threshold, and preserve actual generated outputs. Output: final test table/confusion matrix and limitations.

In [ ]:
# TODO: run once after `selected_threshold` and configuration are approved.
# final_pipeline = clone(selected_pipeline).fit(X_train, y_train)
# test_probability = final_pipeline.predict_proba(X_test)[:, 1]
# test_metrics = classification_metrics(
#     y_test, test_probability, threshold=selected_threshold
# )
# display(pd.Series(test_metrics, name="protected_test"))

## 14. Coefficient/Odds Ratio Interpretation

Interpret standardised numeric coefficients and one-hot categories from the fitted pipeline. State reference-category implications, uncertainty/instability and that coefficients are associations—not causal effects.

**TODO (evaluation owner):** extract transformed feature names and coefficients; calculate odds ratios; review suspicious or unstable values. Output: labelled coefficient table and careful business interpretation.

In [ ]:
# TODO: run after final fit.
# feature_names = final_pipeline.named_steps["preprocessor"].get_feature_names_out()
# coefficients = final_pipeline.named_steps["classifier"].coef_[0]
# coefficient_table = pd.DataFrame({
#     "feature": feature_names,
#     "coefficient": coefficients,
#     "odds_ratio": np.exp(coefficients),
# }).sort_values("coefficient")
# display(coefficient_table)

## 15. Error Analysis

Review false negatives and false positives across operationally relevant cohorts without using the findings to silently tune against the test set. Small subgroup counts require caution.

**TODO (evaluation owner):** join predictions to trace-only `order_id`, compare error slices (for example state, category, late/on-time and value bands), and inspect missingness. Output: cohort table and hypotheses for future validation—not invented explanations.

In [ ]:
# TODO: construct only after final test predictions exist.
# test_errors = X_test.copy()
# test_errors["actual"] = y_test
# test_errors["probability"] = test_probability
# test_errors["predicted"] = (test_probability >= selected_threshold).astype(int)
# Display minimum cohort counts alongside every rate.

## 16. Final Model Decision

**TODO (all):** decide whether the model is fit for the stated customer-experience decision. Compare against dummy, success criteria, variability, error costs and limitations. Output: explicit accept/reject decision and justification. A model need not be accepted merely because it was fitted.

## 17. Production Artifact

The artifact bundles the fitted preprocessing and classifier, frozen threshold, exact feature list, schema version and prediction point. Include provenance metadata. Do not save or commit an unreviewed model.

In [ ]:
ARTIFACT_PATH = PROJECT_ROOT / "deployment" / "negative_review_logistic.joblib"

# TODO: save only after final approval; decide with the team whether binaries are versioned.
# save_inference_artifact(
#     final_pipeline,
#     str(ARTIFACT_PATH),
#     threshold=selected_threshold,
#     metadata={
#         "git_commit": "TODO",
#         "training_data_contract": "Phase 1 order_level.csv + item_level.csv",
#         "selected_by": "TODO",
#     },
# )

## 18. Inference Smoke Test

Reload the persisted bundle in a clean process if practical, score held-out-shaped rows without their labels, and confirm probabilities, classes, unknown-category handling and missing-value handling. This tests the interface, not model quality.

**TODO (evaluation owner):** load with `joblib.load`, call `predict_from_artifact`, and record the actual result or error. Output: reproducible smoke-test evidence.

In [ ]:
# TODO: execute after artifact creation.
# import joblib
# artifact = joblib.load(ARTIFACT_PATH)
# smoke_result = predict_from_artifact(artifact, X_test.iloc[[0]])
# display(smoke_result)

## 19. Phase 3 Handoff

Hand over: artifact path/version, exact input fields and types, prediction point, threshold meaning, training-population eligibility, expected missing/unknown-category behaviour, model limitations and smoke-test instructions. Phase 3 should consume this interface rather than reproduce training transformations.

**TODO (all):** complete the handoff checklist after model approval. The Phase 3 API/UI is intentionally outside this notebook and this foundation PR.